In [1]:
print("Spark:", spark.version)

print(
    "Scala:",
    spark.sparkContext._jvm.scala.util.Properties.versionNumberString()
)

print(
    "Java:",
    spark.sparkContext._jvm.java.lang.System.getProperty("java.version")
)

Spark: 3.5.0
Scala: 2.12.18
Java: 17.0.21.0.3


In [2]:
jvm = spark.sparkContext._jvm

classes = [
    "org.apache.iceberg.spark.SparkCatalog",
    "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    "org.apache.iceberg.rest.RESTCatalog"
]

for class_name in classes:
    try:
        jvm.java.lang.Class.forName(class_name)
        print(f"[OK] {class_name}")
    except Exception as e:
        print(f"[NÃO DISPONÍVEL] {class_name}")

[OK] org.apache.iceberg.spark.SparkCatalog
[OK] org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
[OK] org.apache.iceberg.rest.RESTCatalog


In [3]:
for key, value in spark.sparkContext.getConf().getAll():
    if "iceberg" in key.lower() or "catalog" in key.lower():
        print(f"{key} = {value}")

spark.sql.catalogImplementation = hive
spark.sql.catalog.spark_catalog = org.apache.spark.sql.hive.catalog.CentrifyCatalog
spark.sql.catalog.spark_catalog.defaultDatabase.autoCreate = false
spark.insert.catalog = test


In [4]:
jvm = spark.sparkContext._jvm

classes = [
    "org.apache.iceberg.spark.SparkCatalog",
    "org.apache.iceberg.rest.RESTCatalog",
    "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
]

for name in classes:
    cls = jvm.java.lang.Class.forName(name)
    package = cls.getPackage()

    print(f"\nClasse: {name}")
    print("Versão:", package.getImplementationVersion())
    print("Origem:", cls.getProtectionDomain()
          .getCodeSource().getLocation().toString())


Classe: org.apache.iceberg.spark.SparkCatalog
Versão: None
Origem: file:/opt/spark/ext/classpath/iceberg-spark-runtime-3.5_2.12-1.8.x-oci-1.0.0-8-eea9a9b79884628e23889f1030510255de073940.jar

Classe: org.apache.iceberg.rest.RESTCatalog
Versão: None
Origem: file:/opt/spark/ext/classpath/iceberg-spark-runtime-3.5_2.12-1.8.x-oci-1.0.0-8-eea9a9b79884628e23889f1030510255de073940.jar

Classe: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Versão: None
Origem: file:/opt/spark/ext/classpath/iceberg-spark-runtime-3.5_2.12-1.8.x-oci-1.0.0-8-eea9a9b79884628e23889f1030510255de073940.jar


In [5]:
spark.sql("SHOW NAMESPACES IN oadc").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [6]:
# Variaveis de sessão e organização de credenciais de usuário senhas e sessões.

spark.conf.set(
    "spark.sql.catalog.aicat.s3.access-key-id",
    "9340b73b9fa4150db3d6d85238faf7912a8fac1f"
)

spark.conf.set(
    "spark.sql.catalog.aicat.s3.secret-access-key",
    aidputils.secrets.get(name="SECRET_KEY_OBS", key="SECRET_KEY")
)
 
spark.conf.set(
    "spark.sql.catalog.aicat.credential",
    aidputils.secrets.get(name="ICEBERG_REST_CATALOG", key="USER:PASSWORD")
)

spark.conf.set(
    "spark.sql.catalog.aicat.scope",
    "PRINCIPAL_ROLE:ALL"
)


# Nome lógico do catálogo no Spark
spark.conf.set(
    "spark.sql.catalog.aicat",
    "org.apache.iceberg.spark.SparkCatalog"
)

# Tipo de catálogo
spark.conf.set(
    "spark.sql.catalog.aicat.type",
    "rest"
)

# Endpoint REST do Oracle AI Data Catalog
spark.conf.set(
    "spark.sql.catalog.aicat.uri",
    "https://G84191C2534E979-POCLAKEHOUSEICEBERG.adb.sa-saopaulo-1.oraclecloudapps.com/catalog"
)

# Nome do warehouse do catálogo Iceberg
spark.conf.set(
    "spark.sql.catalog.aicat.warehouse",
    "aicat_iceberg_rest_catalog"
)

# Autenticação OAuth 2.0
spark.conf.set(
    "spark.sql.catalog.aicat.rest.auth.type",
    "oauth2"
)

spark.conf.set(
    "spark.sql.catalog.aicat.oauth2-server-uri",
    "https://G84191C2534E979-POCLAKEHOUSEICEBERG.adb.sa-saopaulo-1.oraclecloudapps.com/catalog/v1/auth/token"
)

In [7]:
spark.sql("""
    SHOW NAMESPACES IN aicat
""").show(truncate=False)

+---------+
|namespace|
+---------+
|ADMIN    |
|BRONZE   |
|GGADMIN  |
|OURO     |
|PRATA    |
+---------+



In [8]:
spark.sql("""
    SHOW TABLES IN aicat.BRONZE
""").show(truncate=False)

+---------+------------+-----------+
|namespace|tableName   |isTemporary|
+---------+------------+-----------+
|BRONZE   |DS_BENEFICIO|false      |
+---------+------------+-----------+



In [9]:
spark.sql("""
    DESCRIBE TABLE aicat.BRONZE.DS_BENEFICIO
""").show(200, truncate=False)

+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|NU_NB     |bigint   |NULL   |
|CS_ESPECIE|bigint   |NULL   |
|D2_DDB    |bigint   |NULL   |
+----------+---------+-------+



In [10]:
spark.sql("""
    SELECT *
    FROM aicat.BRONZE.DS_BENEFICIO
    LIMIT 10
""").show(truncate=False)

+----------+----------+--------+
|NU_NB     |CS_ESPECIE|D2_DDB  |
+----------+----------+--------+
|1000141490|39        |20190821|
|1000141491|96        |20170406|
|1000141492|75        |20190731|
|1000141493|83        |20240204|
|1000141494|96        |20200414|
|1000141484|73        |20130824|
|1000141485|42        |20110815|
|1000141486|97        |20180227|
|1000141487|21        |20150702|
|1000141488|21        |20151012|
+----------+----------+--------+

